# Городища: признаки DINOv2-B по виньеткам SpOr 384/768 м (bot-5)
**Только честный замер. Ничего не идёт в сдачу до письма организаторов.**
Код: `scripts/gor_patch/extract_feats.py`. Метки здесь не используются; оценка (LOFO, П1–П5) — на боксе, `kaggle/README_KAGGLE_GOR.md`.

Inputs: `expds-koz3-bot5` (репо, ветка `bot5-gor-patch-kaggle`), датасет с `train-expds-3` (нужны только `*/*SpOr*/*.tif*` и `*/UTM.json`). Accelerator: GPU T4, Internet ON (скачать веса facebook/dinov2-base, Apache-2.0).

In [ ]:
REPO='/kaggle/input/expds-koz3-bot5'   # поправить на фактический путь
!pip install -q -r $REPO/kaggle/requirements_gor.txt 2>&1 | tail -1
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.device_count())
assert transformers.__version__ == '5.21.0', transformers.__version__

In [ ]:
import os
os.environ.update(DATA='/kaggle/input/train-expds-3', OUT='/kaggle/working/gor', MODEL='facebook/dinov2-base', DEVICE='cuda', TAG='_b', THREADS='4')
os.makedirs(os.environ['OUT'], exist_ok=True)
!ls $DATA | head

## Шаг 1 — smoke на 3 файлах (обязателен первым)

In [ ]:
!LIMIT_FILES=3 python $REPO/scripts/gor_patch/extract_feats.py 2>&1 | tail -5

## Шаг 2 — полный прогон (на CPU-боксе DINOv2-S ≈0,15 с/виньетка; на T4 ожидается быстрее, засечь по логу)

In [ ]:
!python $REPO/scripts/gor_patch/extract_feats.py 2>&1 | tail -3
!cd /kaggle/working/gor && ls -la feats_b.npz meta_b.json && tar czf /kaggle/working/gor_feats_b.tgz feats_b.npz meta_b.json

Скачать `gor_feats_b.tgz` (Output) и передать bot-5: распаковать в `analysis/gor_patch/`, затем на боксе `FEATS_TAG=_b python scripts/gor_patch/gor_patch_eval.py` → `gor_patch_eval_b.json` (П1–П5).